In [0]:
# ══════════════════════════════════════
# 00_DATA_QUALITY_RULES
# Squad 3 — Arquitetura Medalhão
# Regras de Qualidade Versionadas
# Luiz Henrique Portácio
# ══════════════════════════════════════

# Regras de Qualidade de Dados — Squad 3

Este notebook é a fonte única de verdade para as regras de
qualidade aplicadas na camada Silver. Toda alteração de regra deve
ser documentada aqui, com nova versão e data de vigência, antes de
ser implementada nos notebooks Silver.

## Princípio de Governança

**Silver qualifica os dados, Gold decide o que consumir.**

- A Silver nunca exclui registros por problemas de qualidade
  (exceto PK nula, que é erro fatal por comprometer a identidade
  do dado).
- Toda regra de qualidade gera uma flag `flag_<regra>_invalido`
  (`TRUE` = problema, `FALSE` = registro válido para aquela regra).
- Os valores originais são sempre preservados. Quando uma versão
  tratada faz sentido para consumo analítico, ela é criada em uma
  coluna adicional (ex: `cnpj_tratado`, `estado_tratado`), nunca
  sobrescrevendo o dado original.
- A Gold aplica os filtros oficiais documentados no topo de cada
  notebook Gold, decidindo o que entra no dataset analítico.

## Exceções ao princípio (erros fatais)

- **PK nula**: vai para quarentena E interrompe o pipeline
  (`raise Exception`), pois compromete a identidade do dado.
- **PK duplicada**: os registros excedentes (todos exceto o
  vencedor da deduplicação) vão para quarentena, mas o pipeline
  **continua** normalmente com o vencedor.

In [0]:
# ══════════════════════════════════════
# VERSAO ATUAL DAS REGRAS
# ══════════════════════════════════════

DQ_RULES_VERSION = "v1.0"
DQ_RULES_VIGENCIA_DESDE = "2026-06-16"

print(f"Versao de regras de qualidade em uso: {DQ_RULES_VERSION}")
print(f"Vigente desde: {DQ_RULES_VIGENCIA_DESDE}")

### Histórico de Versões

| Versão | Vigência desde | Mudanças |
|---|---|---|
| v1.0 | 2026-06-16 | Versão inicial. Implementa flags de qualidade padronizadas (`flag_<regra>_invalido`), quarentena para PK nula/duplicada, métricas centralizadas de DQ, e tratamento de CNPJ, UF, peso_vendas, quantidade, preço, código_barras e valor_total_item. |
| v1.1 | 2026-06-20 | Adendo: dicionário `PRODUTO_CATEGORIA_MAP` e função `classificar_produto()` para viabilizar o KPI 8 (perecível vs seco). Classificação por inferência sobre o nome da categoria-base do `codigo_barras_produto` — sem validação contra fonte oficial, recomendada revisão humana. |

### Regras — physical_lojas

| Campo | Regra | Tratamento | Flag |
|---|---|---|---|
| id_loja | PK nula | Quarentena + erro fatal (para o pipeline) | n/a |
| id_loja | PK duplicada | Mantém o registro mais recente (`bronze_ingested_at` desc); excedentes vão para quarentena | n/a |
| cnpj | Deve ter 14 dígitos numéricos | Preserva original; cria `cnpj_tratado` = "NAO_INFORMADO" quando inválido | `flag_cnpj_invalido` |
| cnpj | Duplicado | Mantém o registro mais recente; excedentes vão para quarentena | n/a |
| estado_loja | Deve ser UF válida (2 letras) | Normaliza texto, aplica dicionário de correção (nome completo/variações -> sigla); cria `estado_tratado`; se não resolver, `estado_tratado` = "NAO_INFORMADO" | `flag_uf_invalido` |
| nome_loja | Valor "nan" ou nulo | Substitui por "Nao Informado" | n/a (tratamento direto, sem ambiguidade de dado real) |
| cidade_loja | Vazia ou nula | Substitui por "NAO_INFORMADO" | n/a (tratamento direto) |
| peso_vendas | Fora da faixa esperada (calculada via percentis P1/P99 da distribuição real) | Preserva original | `flag_peso_vendas_invalido` |

## Regras — physical_itens_venda_caixa

| Campo | Regra | Tratamento | Flag |
|---|---|---|---|
| id_item_venda | PK nula | Quarentena + erro fatal (para o pipeline) | n/a |
| id_item_venda | PK duplicada | Mantém o registro mais recente; excedentes vão para quarentena | n/a |
| id_transacao | FK órfã (não existe em physical_vendas_caixa) | Preserva o registro | `flag_fk_invalido` |
| quantidade | <= 0 | Preserva original | `flag_quantidade_invalido` |
| preco_unitario_registro | <= 0 | Preserva original | `flag_preco_invalido` |
| valor_total_item | Inconsistente com preço * quantidade (tolerância R$ 0.05) | Cria `valor_total_item_original`, `valor_calculado`, `diferenca_valor`, `valor_item_analitico` | `flag_valor_inconsistente` |
| codigo_barras_produto | Vazio ou nulo | Preserva original | `flag_codigo_barras_ausente` |
| codigo_barras_produto | Classificação perecível/seco (v1.1) | Cria `categoria_produto` via `classificar_produto()` | n/a (usado pelo KPI 8 na Gold) |

## Coluna `valor_item_analitico` — regra de cálculo

```
valor_item_analitico = valor_calculado
(sempre o preco_unitario_registro * quantidade,
 independente de flag_valor_inconsistente)
```

Esta é a única coluna que a Gold deve usar para cálculos de receita
e demais KPIs financeiros. `valor_total_item_original` fica
disponível apenas para auditoria e investigação.

In [0]:
# ══════════════════════════════════════
# DICIONARIO DE CORRECAO DE UF
# ══════════════════════════════════════

UF_CORRECTION_MAP = {
    # Siglas (identidade — garante que uma sigla já válida passe direto)
    "AC": "AC", "AL": "AL", "AP": "AP", "AM": "AM", "BA": "BA",
    "CE": "CE", "DF": "DF", "ES": "ES", "GO": "GO", "MA": "MA",
    "MT": "MT", "MS": "MS", "MG": "MG", "PA": "PA", "PB": "PB",
    "PR": "PR", "PE": "PE", "PI": "PI", "RJ": "RJ", "RN": "RN",
    "RS": "RS", "RO": "RO", "RR": "RR", "SC": "SC", "SP": "SP",
    "SE": "SE", "TO": "TO",

    # Nomes completos (sem acento, maiusculo) -> sigla
    "ACRE": "AC",
    "ALAGOAS": "AL",
    "AMAPA": "AP",
    "AMAZONAS": "AM",
    "BAHIA": "BA",
    "CEARA": "CE",
    "DISTRITO FEDERAL": "DF",
    "ESPIRITO SANTO": "ES",
    "GOIAS": "GO",
    "MARANHAO": "MA",
    "MATO GROSSO": "MT",
    "MATO GROSSO DO SUL": "MS",
    "MINAS GERAIS": "MG",
    "PARA": "PA",
    "PARAIBA": "PB",
    "PARANA": "PR",
    "PERNAMBUCO": "PE",
    "PIAUI": "PI",
    "RIO DE JANEIRO": "RJ",
    "RIO GRANDE DO NORTE": "RN",
    "RIO GRANDE DO SUL": "RS",
    "RONDONIA": "RO",
    "RORAIMA": "RR",
    "SANTA CATARINA": "SC",
    "SAO PAULO": "SP",
    "SERGIPE": "SE",
    "TOCANTINS": "TO",

    # Variacoes comuns de digitacao (acentos preservados,
    # caso o texto chegue com acentuacao antes da normalizacao)
    "ACRE ": "AC",
    "AMAPÁ": "AP",
    "CEARÁ": "CE",
    "ESPÍRITO SANTO": "ES",
    "GOIÁS": "GO",
    "MARANHÃO": "MA",
    "PARÁ": "PA",
    "PARAÍBA": "PB",
    "PARANÁ": "PR",
    "PIAUÍ": "PI",
    "RONDÔNIA": "RO",
    "SÃO PAULO": "SP",

    # Erros comuns de digitacao / abreviacao incorreta
    "SP ": "SP",
    "S.P.": "SP",
    "S P": "SP",
    "RJ ": "RJ",
    "R.J.": "RJ",
    "MG ": "MG",
    "M.G.": "MG",
}

print(f"Dicionario de correção de UF carregado: {len(UF_CORRECTION_MAP)} entradas")

In [0]:
# ══════════════════════════════════════
# FUNCAO DE NORMALIZACAO E CORREÇÃO DE UF
# ══════════════════════════════════════

def normalize_text_for_uf_lookup(value):
    """
    Normaliza texto para busca no dicionario de UF:
    remove acentos, espacos extras, converte para maiusculo.
    """
    import unicodedata

    if value is None:
        return None

    texto = str(value).strip().upper()
    texto_sem_acento = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    return texto_sem_acento.strip()


def correct_uf(value):
    """
    Tenta corrigir um valor de UF usando o dicionario de
    correcao. Retorna a sigla corrigida, ou "NAO_INFORMADO"
    se nao for possivel resolver.
    """
    if value is None:
        return "NAO_INFORMADO"

    valor_normalizado = normalize_text_for_uf_lookup(value)

    # tenta direto no dicionario (cobre siglas e nomes com acento)
    if value.strip().upper() in UF_CORRECTION_MAP:
        return UF_CORRECTION_MAP[value.strip().upper()]

    # tenta a versao normalizada (sem acento)
    if valor_normalizado in UF_CORRECTION_MAP:
        return UF_CORRECTION_MAP[valor_normalizado]

    # nao foi possivel resolver
    return "NAO_INFORMADO"


# teste rapido da funcao
testes = ["SP", "sao paulo", "São Paulo", "RIO DE JANEIRO",
          "rj", "Acre", "XYZ", None, "minas gerais"]

print("Teste da funcao correct_uf():")
for teste in testes:
    print(f"   {str(teste):<20} -> {correct_uf(teste)}")

## Classificação Perecível vs Seco (KPI 8 — physical_itens_venda_caixa)

**Adendo à versão v1.0 das regras**, incorporado para viabilizar o
KPI 8 ("Crescimento MoM de perecíveis vs secos por loja"), antes
marcado como não implementado por falta de fonte de classificação.

**Origem da classificação:** o campo `codigo_barras_produto` em
`physical_itens_venda_caixa` não é um código de barras EAN/GTIN
tradicional — é um slug descritivo no formato
`<categoria_produto>-<marca>-<peso/volume>-<variante>`
(ex.: `queijosfatiadospecas-seara-015kg-minasfrescal`). O primeiro
segmento (antes do primeiro `-`) identifica a categoria do produto
de forma consistente em toda a base.

O dicionário `PRODUTO_CATEGORIA_MAP` abaixo foi construído
analisando as 128 categorias-base distintas presentes na base real
de `physical_itens_venda_caixa.csv` (1.565 produtos distintos,
cobertura de 100% das categorias encontradas).

**Critério de classificação:**
- `perecivel`: produtos que exigem refrigeração/congelamento ou têm
  validade curta (laticínios, carnes, frios, hortifruti, padaria
  fresca, congelados).
- `seco`: produtos de mercearia seca, limpeza, higiene e bebidas
  não-refrigeradas, com validade longa em temperatura ambiente.

**⚠️ Limitação importante — revisão humana recomendada:** esta
classificação foi feita por inferência a partir do nome da
categoria, sem validação contra uma fonte oficial de dados (ao
contrário do `UF_CORRECTION_MAP`, que se baseia na lista oficial e
finita de 27 UFs do Brasil). Casos de borda existem — por exemplo,
`cremedeleiteuht` e `leitesvegetaisuht` foram classificados como
`perecivel` por convenção de prateleira refrigerada de mercado,
ainda que sejam produtos UHT de longa vida. Recomenda-se que a
squad revise esta lista antes de considerá-la definitiva para
decisões de negócio.

In [0]:
# ══════════════════════════════════════
# CLASSIFICACAO PERECIVEL x SECO
# Por categoria-base extraida do
# codigo_barras_produto (slug)
# ══════════════════════════════════════

PRODUTO_CATEGORIA_MAP = {
    "absorventes": "seco",
    "achocolatados": "seco",
    "acucaradocantes": "seco",
    "aguamineralcomesemgas": "seco",
    "aguasanitaria": "seco",
    "amaciantes": "seco",
    "amendoim": "seco",
    "areiasanitaria": "seco",
    "arroz": "seco",
    "atumsardinhaenlatados": "seco",
    "aveia": "seco",
    "azeite": "seco",
    "balasgomas": "seco",
    "barbearialaminasespuma": "seco",
    "batatarede": "perecivel",
    "biscoitosdocessalgados": "seco",
    "bolosindustrializados": "seco",
    "bolostortasfresco": "perecivel",
    "cafepograocapsula": "seco",
    "castanhasnozespacote": "seco",
    "cebolarede": "perecivel",
    "cereaisgranolas": "seco",
    "cervejaslatagarrafa": "seco",
    "chasprontosgarrafa": "seco",
    "chassaches": "seco",
    "chocolatesbarrasbombons": "seco",
    "compostolacteo": "seco",
    "condicionadores": "seco",
    "conservasazeitonapalmitopicles": "seco",
    "copospratosdescartaveis": "seco",
    "cortesbovinos": "perecivel",
    "cortesdefrango": "perecivel",
    "cortessuinos": "perecivel",
    "cremedeavela": "seco",
    "cremedeleiteuht": "perecivel",
    "cremedental": "seco",
    "desinfetantes": "seco",
    "desodorantes": "seco",
    "destiladoswhiskyginvodkacachaca": "seco",
    "detergentes": "seco",
    "docespotescompotas": "seco",
    "energeticos": "seco",
    "enlatadosmilhoervilhaseleta": "seco",
    "enxaguantebucal": "seco",
    "escovasdedente": "seco",
    "esponjaslasdeaco": "seco",
    "espumantes": "seco",
    "farinaceosfubatapiocamandioca": "seco",
    "farinhadetrigo": "seco",
    "feijao": "seco",
    "filmepvc": "seco",
    "filtrosdecafe": "seco",
    "fiodental": "seco",
    "formulasinfantisleiteempo": "seco",
    "fraldas": "seco",
    "frutas": "perecivel",
    "frutassecasuvapassadamasco": "seco",
    "gelatinaspudinspo": "seco",
    "geleias": "seco",
    "guardanapos": "seco",
    "higieneinfantilshampoosabonete": "seco",
    "higienepetshampoolencos": "seco",
    "inseticidas": "seco",
    "iogurtes": "perecivel",
    "lavaloucasmaquina": "seco",
    "legumes": "perecivel",
    "leiteempo": "seco",
    "leitefresco": "perecivel",
    "leitesvegetaisuht": "perecivel",
    "leiteuht": "seco",
    "lencosumedecidos": "seco",
    "licores": "seco",
    "limpadoresespecificosvidropiso": "seco",
    "linguicasalsicha": "perecivel",
    "macarraomassassecas": "seco",
    "maioneseketchupmostarda": "seco",
    "manteigamargarina": "perecivel",
    "misturasparabolo": "seco",
    "molhostomatepestouht": "seco",
    "mortadelasalame": "perecivel",
    "multiuso": "seco",
    "nectares": "seco",
    "oleogorduras": "seco",
    "ovos": "perecivel",
    "paesdeforma": "seco",
    "paesproducaopropria": "perecivel",
    "panetonescolombassazonal": "seco",
    "paodequeijocongelado": "perecivel",
    "papelaluminio": "seco",
    "papelhigienico": "seco",
    "papeltoalha": "seco",
    "papinhaspotes": "perecivel",
    "peixesfrutosdomar": "perecivel",
    "petiscos": "seco",
    "pilhasbaterias": "seco",
    "pipocamicroondasgrao": "seco",
    "polpasdefruta": "perecivel",
    "pomadasparaassadura": "seco",
    "pratosprontoscongelados": "perecivel",
    "presuntopeitodeperu": "perecivel",
    "protetorsolar": "seco",
    "purificadoresdear": "seco",
    "queijosfatiadospecas": "perecivel",
    "racaocaes": "seco",
    "racaogatos": "seco",
    "refrigerantes": "seco",
    "requeijao": "perecivel",
    "sabaopoliquido": "seco",
    "sabonetesbarraliquido": "seco",
    "sacosdelixo": "seco",
    "sal": "seco",
    "shampoos": "seco",
    "snackssalgadinhos": "seco",
    "sopascremespoinstantaneo": "seco",
    "sorvetes": "perecivel",
    "sucosempo": "seco",
    "sucosprontoscaixa": "seco",
    "talheresdescartaveis": "seco",
    "tapeteshigienicos": "seco",
    "temperoscaldos": "seco",
    "torradas": "seco",
    "tratamentoscapilares": "seco",
    "utensiliosdecozinhasimples": "seco",
    "vegetaiscongelados": "perecivel",
    "velas": "seco",
    "verduras": "perecivel",
    "vinagre": "seco",
    "vinhostintobrancorose": "seco",
    "algodao": "seco",
    "fosforosacendedores": "seco",
    "hastesflexiveis": "seco",
    "lampadas": "seco",
}

print(f"Dicionario de classificacao perecivel/seco carregado: {len(PRODUTO_CATEGORIA_MAP)} categorias-base")

In [0]:
# ══════════════════════════════════════
# FUNCAO DE CLASSIFICACAO PERECIVEL x SECO
# ══════════════════════════════════════

def classificar_produto(codigo_barras_produto):
    """
    Classifica um produto como "perecivel" ou "seco" com base na
    categoria-base extraida do codigo_barras_produto (primeiro
    segmento antes do primeiro "-").

    Retorna "NAO_CLASSIFICADO" se a categoria-base nao estiver no
    dicionario PRODUTO_CATEGORIA_MAP (produto novo, fora da cobertura
    conhecida na data de construcao deste dicionario).
    """
    if codigo_barras_produto is None:
        return "NAO_CLASSIFICADO"

    categoria_base = str(codigo_barras_produto).strip().lower().split("-")[0]
    return PRODUTO_CATEGORIA_MAP.get(categoria_base, "NAO_CLASSIFICADO")


# teste rapido da funcao
testes_produto = [
    "queijosfatiadospecas-seara-015kg-minasfrescal",
    "arroz-tio joao-5kg-tipo1",
    "ovos-granjafeliz-30un-vermelhocaipira12un",
    "produtoinexistente-marca-1un",
    None,
]

print("Teste da funcao classificar_produto():")
for teste in testes_produto:
    print(f"   {str(teste):<55} -> {classificar_produto(teste)}")

In [0]:
# Resumo

print("=" * 55)
print("00_data_quality_rules carregado com sucesso!")
print("=" * 55)
print(f"""
Versao das regras  : {DQ_RULES_VERSION}
Vigente desde       : {DQ_RULES_VIGENCIA_DESDE}

Disponivel para uso nos notebooks Silver:
   DQ_RULES_VERSION         -> versao atual das regras
   UF_CORRECTION_MAP         -> dicionario de correcao de UF
   normalize_text_for_uf_lookup() -> normaliza texto p/ UF
   correct_uf()              -> corrige/valida UF
   PRODUTO_CATEGORIA_MAP     -> dicionario perecivel/seco (v1.1)
   classificar_produto()     -> classifica produto em perecivel/seco

Uso nos notebooks Silver:
   %run "../governanca/00_data_quality_rules"
""")